In [0]:
# read the bronze stream
bronze_stream = (
    spark.readStream
        .format("delta")
#        .load("/Volumes/ingestion/raw_ads/raw/delta/ads_bronze")
        .load("/Volumes/ingestion/raw_ads/raw/delta/ads_bronze_file")
)

# define event schema
from pyspark.sql.types import *

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_type", StringType()),
    StructField("event_time", StringType()),
    StructField("campaign_id", StringType()),
    StructField("user_id", StringType()),
    StructField("session_id", StringType()),
    StructField("geo", StringType()),
    StructField("device", StringType()),
    StructField("price_paid", DoubleType()),
    StructField("conversion_value", DoubleType())
])

# parse json
from pyspark.sql.functions import from_json, col

silver_df = bronze_stream.select(
    from_json(col("raw_json"), event_schema).alias("data"),
    col("kafka_timestamp")
).select("data.*", "kafka_timestamp")

# dedup events
silver_df = (
    silver_df
    .withWatermark("kafka_timestamp", "10 minutes")
    .dropDuplicates(["event_id"])
)

# normalize event time
from pyspark.sql.functions import to_timestamp

silver_df = silver_df.withColumn(
    "event_time",
    to_timestamp("event_time")
)

# write silver table
silver_query = (
    silver_df.writeStream
        .format("delta")
        .outputMode("append")
        #.option("checkpointLocation", "/Volumes/ingestion/raw_ads/raw/checkpoints/ads_silver")
        .option("checkpointLocation", "/Volumes/ingestion/raw_ads/raw/checkpoints/ads_silver_file")
        .outputMode("append")
        #.start("/Volumes/ingestion/raw_ads/raw/delta/ads_silver")
        .start("/Volumes/ingestion/raw_ads/raw/delta/ads_silver_file")
)